# Validation Notebook

This notebook validates data integrity across Bronze → Silver → Gold layers.
It produces a structured report and stops on critical failures.

## Check Summary
- Source → Bronze row counts and data equality
- Silver brand/manufacturer quality
- Silver primary/composite key uniqueness
- Foreign key referential integrity
- Bronze → Silver part reconciliation
- Silver → Gold line count preservation
- Revenue reconciliation across layers
- Monthly vs detailed Gold totals
- Market share validity (0–1, sum to 1)
- Gold unit margin correctness
- Price deviation product accuracy

The final section demonstrates that checks catch deliberately introduced errors.

In [0]:
from pyspark.sql import Row
from datetime import datetime
import uuid

run_id = str(uuid.uuid4())[:8]
checked_at = datetime.now().isoformat()
results = []

def run_check(check_name, layer, expected, query, is_critical=True):
    """Run a validation check and record the result."""
    df = spark.sql(query)
    rows = df.collect()
    actual = rows[0][0] if rows else None
    passed = (actual == expected) if expected is not None else (actual is not None)
    details = f"expected={expected}, actual={actual}"
    if not passed and rows and len(rows) > 0:
        details += f" | full_row={rows[0].asDict()}"
    results.append(Row(
        run_id=run_id,
        checked_at=checked_at,
        check_name=check_name,
        layer=layer,
        expected_value=str(expected),
        actual_value=str(actual),
        passed=passed,
        details=details,
        is_critical=is_critical
    ))
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {check_name}: {details}")
    return passed

def run_check_query(check_name, layer, query, is_critical=True):
    """Run a check that should return 0 (no violations)."""
    return run_check(check_name, layer, 0, query, is_critical)

print(f"Validation run_id: {run_id}")
print(f"Checked at: {checked_at}\n")

In [0]:
print("=== BRONZE LAYER CHECKS ===\n")

tpch_tables = ["customer", "lineitem", "nation", "orders", "part", "partsupp", "region", "supplier"]

# Row count checks: source vs Bronze
for table in tpch_tables:
    src_count = spark.sql(f"SELECT COUNT(*) FROM samples.tpch.{table}").collect()[0][0]
    run_check(
        f"Bronze row count: {table}",
        "Bronze",
        src_count,
        f"SELECT COUNT(*) FROM bda_bronze.{table}"
    )

# Data equality (EXCEPT ALL both directions) for key tables
for table in ["part", "orders", "partsupp"]:
    cols = spark.sql(f"SHOW COLUMNS FROM bda_bronze.{table}").collect()
    col_names = [c.col_name for c in cols if c.col_name != "ingestion_timestamp"]
    col_list = ", ".join(col_names)

    run_check(f"Bronze data equality (src→bronze): {table}", "Bronze", 0,
        f"SELECT COUNT(*) FROM (SELECT {col_list} FROM samples.tpch.{table} EXCEPT ALL SELECT {col_list} FROM bda_bronze.{table})")
    run_check(f"Bronze data equality (bronze→src): {table}", "Bronze", 0,
        f"SELECT COUNT(*) FROM (SELECT {col_list} FROM bda_bronze.{table} EXCEPT ALL SELECT {col_list} FROM samples.tpch.{table})")

print()

In [0]:
print("=== SILVER LAYER CHECKS ===\n")

# Brand/manufacturer quality
run_check_query("Silver part: null brands", "Silver",
    "SELECT COUNT(*) FROM bda_silver.part WHERE p_brand IS NULL")
run_check_query("Silver part: empty/whitespace brands", "Silver",
    "SELECT COUNT(*) FROM bda_silver.part WHERE TRIM(p_brand) = ''")
run_check_query("Silver part: null manufacturers", "Silver",
    "SELECT COUNT(*) FROM bda_silver.part WHERE p_mfgr IS NULL")
run_check_query("Silver part: empty/whitespace manufacturers", "Silver",
    "SELECT COUNT(*) FROM bda_silver.part WHERE TRIM(p_mfgr) = ''")

# Primary key uniqueness
run_check_query("Silver part: duplicate p_partkey", "Silver",
    "SELECT COUNT(*) FROM (SELECT p_partkey, COUNT(*) AS c FROM bda_silver.part GROUP BY p_partkey HAVING c > 1)")
run_check_query("Silver orders: duplicate o_orderkey", "Silver",
    "SELECT COUNT(*) FROM (SELECT o_orderkey, COUNT(*) AS c FROM bda_silver.orders GROUP BY o_orderkey HAVING c > 1)")
run_check_query("Silver partsupp: duplicate (ps_partkey, ps_suppkey)", "Silver",
    "SELECT COUNT(*) FROM (SELECT ps_partkey, ps_suppkey, COUNT(*) AS c FROM bda_silver.partsupp GROUP BY ps_partkey, ps_suppkey HAVING c > 1)")
run_check_query("Silver lineitem: duplicate (l_orderkey, l_linenumber)", "Silver",
    "SELECT COUNT(*) FROM (SELECT l_orderkey, l_linenumber, COUNT(*) AS c FROM bda_silver.lineitem GROUP BY l_orderkey, l_linenumber HAVING c > 1)")

# Foreign key checks
run_check_query("Silver lineitem→partsupp orphans", "Silver",
    """SELECT COUNT(*) FROM bda_silver.lineitem l
       LEFT ANTI JOIN bda_silver.partsupp ps
       ON l.l_partkey = ps.ps_partkey AND l.l_suppkey = ps.ps_suppkey""")
run_check_query("Silver lineitem→orders orphans", "Silver",
    """SELECT COUNT(*) FROM bda_silver.lineitem l
       LEFT ANTI JOIN bda_silver.orders o
       ON l.l_orderkey = o.o_orderkey""")
run_check_query("Silver lineitem→part orphans", "Silver",
    """SELECT COUNT(*) FROM bda_silver.lineitem l
       LEFT ANTI JOIN bda_silver.part p
       ON l.l_partkey = p.p_partkey""")

# Positive quantity and list price
run_check_query("Silver part: non-positive retailprice", "Silver",
    "SELECT COUNT(*) FROM bda_silver.part WHERE p_retailprice <= 0")
run_check_query("Silver lineitem: non-positive quantity", "Silver",
    "SELECT COUNT(*) FROM bda_silver.lineitem WHERE l_quantity <= 0")

print()

In [0]:
print("=== RECONCILIATION CHECKS ===\n")

# Bronze → Silver part reconciliation
bronze_part_count = spark.sql("SELECT COUNT(*) FROM bda_bronze.part").collect()[0][0]
silver_part_count = spark.sql("SELECT COUNT(*) FROM bda_silver.part").collect()[0][0]
quarantine_count = spark.sql("SELECT COUNT(*) FROM bda_silver.part_quarantine").collect()[0][0]
run_check("Bronze→Silver: accepted + quarantined = total parts", "Silver→Gold",
    bronze_part_count,
    f"SELECT {silver_part_count} + {quarantine_count}")

# Silver → Gold line count (inner joins should preserve all lines if no orphans)
silver_line_count = spark.sql("SELECT COUNT(*) FROM bda_silver.lineitem").collect()[0][0]
run_check("Silver→Gold: lineitem count = brand_sales count", "Gold",
    silver_line_count,
    "SELECT COUNT(*) FROM bda_gold.brand_sales")

# Revenue reconciliation: Silver lineitem revenue vs Gold brand_sales net_revenue
silver_revenue = float(spark.sql("SELECT SUM(revenue) FROM bda_silver.lineitem").collect()[0][0] or 0)
gold_revenue = float(spark.sql("SELECT SUM(net_revenue) FROM bda_gold.brand_sales").collect()[0][0] or 0)
tolerance = 0.01
rev_diff = abs(silver_revenue - gold_revenue)
run_check(f"Revenue: Silver revenue = Gold net_revenue (tol={tolerance})", "Gold",
    True,
    f"SELECT {rev_diff <= tolerance}",
    is_critical=True)

# Monthly Gold totals = detailed Gold totals
monthly_total = float(spark.sql("SELECT SUM(brand_revenue) FROM bda_gold.brand_category_monthly").collect()[0][0] or 0)
detail_total = float(spark.sql("SELECT SUM(net_revenue) FROM bda_gold.brand_sales").collect()[0][0] or 0)
month_diff = abs(monthly_total - detail_total)
run_check(f"Gold: monthly totals = detailed totals (tol={tolerance})", "Gold",
    True,
    f"SELECT {month_diff <= tolerance}")

print()

In [0]:
print("=== MARKET SHARE & MARGIN CHECKS ===\n")

# Market shares between 0 and 1
run_check_query("Gold: market_share < 0", "Gold",
    "SELECT COUNT(*) FROM bda_gold.brand_category_monthly WHERE market_share < 0")
run_check_query("Gold: market_share > 1", "Gold",
    "SELECT COUNT(*) FROM bda_gold.brand_category_monthly WHERE market_share > 1")

# Category market shares sum to ~1
run_check("Gold: category market shares sum to ~1", "Gold", 0,
    """SELECT COUNT(*) FROM (
        SELECT month_start, category, ROUND(SUM(market_share), 4) AS s
        FROM bda_gold.brand_category_monthly
        GROUP BY month_start, category
        HAVING ROUND(SUM(market_share), 4) NOT BETWEEN 0.99 AND 1.01
    )""")

# Gold unit margin = list price - supply cost
run_check_query("Gold: list_unit_margin != list_unit_price - supply_unit_cost", "Gold",
    """SELECT COUNT(*) FROM bda_gold.brand_sales
       WHERE list_unit_margin != list_unit_price - supply_unit_cost""")

# Price deviation products = distinct products satisfying rule
pd_view_count = spark.sql("SELECT COUNT(*) FROM bda_gold.price_deviation_products").collect()[0][0]
pd_direct_count = spark.sql("SELECT COUNT(DISTINCT part_key) FROM bda_gold.brand_sales WHERE price_deviation_ratio > 0.15").collect()[0][0]
run_check("Gold: price_deviation view count = direct count", "Gold",
    pd_direct_count,
    f"SELECT {pd_view_count}")

print()

In [0]:
print("=== VALIDATION REPORT ===\n")

report_df = spark.createDataFrame(results)
display(report_df)

passed = sum(1 for r in results if r.passed)
failed = sum(1 for r in results if not r.passed)
critical_failed = sum(1 for r in results if not r.passed and r.is_critical)

print(f"\nTotal checks: {len(results)}")
print(f"Passed: {passed}")
print(f"Failed: {failed}")
print(f"Critical failures: {critical_failed}")

if critical_failed > 0:
    print("\n*** CRITICAL FAILURES DETECTED — pipeline should stop ***")
    print("Failed critical checks:")
    for r in results:
        if not r.passed and r.is_critical:
            print(f"  - {r.check_name}: {r.details}")
else:
    print("\n✓ All critical checks passed.")

## Error Demonstration

The following cells create temporary test data with deliberate errors and show that the validation checks catch them.
This proves the checks actually detect problems — a passing clean dataset only shows the current data is valid.

**Test errors:**
1. A part with an empty brand
2. A duplicate partsupp pair
3. A line item with a nonexistent product–supplier pair

In [0]:
print("=== ERROR DEMONSTRATION ===\n")
print("Creating temporary test data with deliberate errors...\n")

# Create temporary test schema
spark.sql("CREATE SCHEMA IF NOT EXISTS validation_test")

# 1. Part with empty brand
spark.sql("""
CREATE OR REPLACE TABLE validation_test.bad_part AS
SELECT * FROM bda_silver.part LIMIT 100
UNION ALL
SELECT
    999999 AS p_partkey, 'BAD PART' AS p_name, 'Manufacturer#1' AS p_mfgr,
    '   ' AS p_brand, 'BAD TYPE' AS p_type, 1 AS p_size,
    'SM CASE' AS p_container, 1000.00 AS p_retailprice,
    'test bad part' AS p_comment, current_timestamp() AS ingestion_timestamp
""")

# 2. Duplicate partsupp pair
spark.sql("""
CREATE OR REPLACE TABLE validation_test.bad_partsupp AS
SELECT * FROM bda_silver.partsupp LIMIT 100
UNION ALL
SELECT * FROM bda_silver.partsupp LIMIT 1
""")

# 3. Line item with nonexistent product-supplier pair
spark.sql("""
CREATE OR REPLACE TABLE validation_test.bad_lineitem AS
SELECT * FROM bda_silver.lineitem LIMIT 100
UNION ALL
SELECT
    1 AS l_orderkey, 1 AS l_partkey, 999999 AS l_suppkey,
    1 AS l_linenumber, 1 AS l_quantity, 1000.00 AS l_extendedprice,
    0 AS l_discount, 0 AS l_tax, 'N' AS l_returnflag,
    'O' AS l_linestatus, current_date() AS l_shipdate,
    current_date() AS l_commitdate, current_date() AS l_receiptdate,
    'NONE' AS l_shipinstruct, 'REG AIR' AS l_shipmode,
    'test bad line' AS l_comment, current_timestamp() AS ingestion_timestamp,
    1000.00 AS revenue, 500.00 AS margin
""")

# Run checks against bad data — they should FAIL
print("--- Check 1: Empty brand in bad_part ---")
bad_brand_count = spark.sql("SELECT COUNT(*) FROM validation_test.bad_part WHERE TRIM(p_brand) = ''").collect()[0][0]
print(f"  Empty brands found: {bad_brand_count} (expected: > 0 → FAIL)")
assert bad_brand_count > 0, "Check failed to detect empty brand!"

print("\n--- Check 2: Duplicate partsupp pair in bad_partsupp ---")
dup_count = spark.sql("""
    SELECT COUNT(*) FROM (
        SELECT ps_partkey, ps_suppkey, COUNT(*) AS c
        FROM validation_test.bad_partsupp
        GROUP BY ps_partkey, ps_suppkey HAVING c > 1
    )
""").collect()[0][0]
print(f"  Duplicate pairs found: {dup_count} (expected: > 0 → FAIL)")
assert dup_count > 0, "Check failed to detect duplicate partsupp!"

print("\n--- Check 3: Orphan lineitem in bad_lineitem ---")
orphan_count = spark.sql("""
    SELECT COUNT(*) FROM validation_test.bad_lineitem l
    LEFT ANTI JOIN bda_silver.partsupp ps
    ON l.l_partkey = ps.ps_partkey AND l.l_suppkey = ps.ps_suppkey
""").collect()[0][0]
print(f"  Orphan lineitems found: {orphan_count} (expected: > 0 → FAIL)")
assert orphan_count > 0, "Check failed to detect orphan lineitem!"

print("\n*** All error demonstrations confirmed: checks detect the introduced problems. ***")

# Cleanup
spark.sql("DROP SCHEMA IF EXISTS validation_test CASCADE")
print("\nTest schema cleaned up.")